In [2]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Set display options to show all rows and columns with full width
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)  # Show full content of each cell
df = pd.read_parquet('../03_processed_datasets/reciprocity_model_7d_799696_users.parquet')
print(df.columns.tolist())

['event_id', 'phase', 'user_id', 'timestamp', 'event', 'question_id', 'phase_one_start', 'phase_two_end', 'event_history', 'is_history', 'has_answer', 'has_accepted_answer', 'time_to_first_answer_hours', 'time_to_accepted_answer_hours', 'time_to_accept_vote_hours', 'numHelped', 'hasAnswer', 'numHelpProvidedEver', 'receivedHelpEver', 'receivedAnswerEver', 'receivedAcceptedAnswerEver', 'receivedAcceptedVoteEver', 'year', 'month', 'numQuestionsAskedAT', 'numHelpReceivedAT', 'numHelpProvidedAT', 'numAnswersReceivedAT', 'numAcceptedAnswersReceivedAT', 'numAcceptedVotesReceivedAT', 'numQuestionsAsked30D', 'numHelpReceived30D', 'numHelpProvided30D', 'numAnswersReceived30D', 'numAcceptedAnswersReceived30D', 'numAcceptedVotesReceived30D', 'numQuestionsAsked14D', 'numHelpReceived14D', 'numHelpProvided14D', 'numAnswersReceived14D', 'numAcceptedAnswersReceived14D', 'numAcceptedVotesReceived14D', 'numQuestionsAsked7D', 'numHelpReceived7D', 'numHelpProvided7D', 'numAnswersReceived7D', 'numAcceptedAn

In [3]:
print(len(df))
df = df[~((df["time_to_first_answer_hours"]>7*24) & (df["has_answer"]==1))]
print(len(df))

7605678
7032848


 # Main Models

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
from IPython.display import HTML, display
from stargazer.stargazer import Stargazer
import sys
import pdfkit
import os
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Create binned variable for numHelpProvidedAT
bins = [0, 1, 2, 3, 4, 5, float('inf')]
labels = ['0', '1', '2', '3', '4', '5+']  # 6 labels for 7 bin edges
df['numHelpProvidedAT_binned'] = pd.cut(df['numHelpProvidedAT'], bins=bins, labels=labels, right=False)
print(f"Created numHelpProvidedAT_binned with distribution:\n{df['numHelpProvidedAT_binned'].value_counts(dropna=False)}")

# Collect all HTML content to combine at the end
all_html_content = []

# Define the function to run models with different dependent variables
def run_models_for_dependent_var(dep_var):
    print(f"\n{'='*50}")
    print(f"Running models for {dep_var}")
    print(f"{'='*50}")

    # Define base formulas
    base_formula = f"{dep_var} ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
    user_fe_formula = f"user_fe_{dep_var} ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
    question_fe_formula = f"question_fe_{dep_var} ~ C(phase) + C(phase):C(has_answer)"

    # Define binned interaction and controls
    binned_interaction = "+ C(numHelpProvidedAT_binned) + C(phase):C(numHelpProvidedAT_binned) + C(has_answer):C(numHelpProvidedAT_binned) + C(phase):C(has_answer):C(numHelpProvidedAT_binned)"
    controls = "+ numQuestionsAskedAT + numHelpReceivedAT + numHelpProvidedAT"

    # Group 1: No fixed effects
    formulas_group1 = [
        base_formula,  # Base model
        base_formula + binned_interaction,  # Base model with binned interactions
        base_formula + binned_interaction + controls,  # Base model with binned interactions and controls
    ]

    # Group 2: User fixed effects
    formulas_group2 = [
        user_fe_formula,  # User FE model
        user_fe_formula + binned_interaction,  # User FE model with binned interactions
        user_fe_formula + binned_interaction + controls,  # User FE model with binned interactions and controls
    ]

    # Group 3: Question fixed effects
    formulas_group3 = [
        question_fe_formula,  # Question FE model
        question_fe_formula + binned_interaction,  # Question FE model with binned interactions
        question_fe_formula + binned_interaction + controls,  # Question FE model with binned interactions and controls
    ]

    all_formula_groups = [formulas_group1, formulas_group2, formulas_group3]
    group_names = ["No Fixed Effects", "User Fixed Effects", "Question Fixed Effects"]

    model_names = [
        ["Base", "Base + Binned", "Base + Binned + Controls"],
        ["User FE", "User FE + Binned", "User FE + Binned + Controls"],
        ["Question FE", "Question FE + Binned", "Question FE + Binned + Controls"]
    ]

    # Run each group of models
    for group_idx, formula_group in enumerate(all_formula_groups):
        print(f"\nRunning {group_names[group_idx]} models...")

        # Fit the models
        models = []
        for formula in formula_group:
            try:
                print(f"Fitting model with formula: {formula}")
                # Fit the model
                model = smf.ols(formula=formula, data=df).fit()

                # Apply clustered standard errors by user_id
                model = model.get_robustcov_results(
                    cov_type='cluster',
                    groups=df['user_id']
                )

                models.append(model)
                print(f"Model fitted successfully")
            except Exception as e:
                print(f"Error fitting model with formula: {formula}")
                print(f"Error: {e}")
                models.append(None)

        # Filter out None models
        models = [m for m in models if m is not None]
        if not models:
            print("No models were successfully fitted in this group.")
            continue

        # Create and customize the Stargazer table
        stargazer = Stargazer(models)
        stargazer.title(f"Effect of Receiving Answers on Providing Help - {dep_var} - {group_names[group_idx]}")
        stargazer.custom_columns(model_names[group_idx][:len(models)], [1] * len(models))
        stargazer.significant_digits(3)
        stargazer.show_degrees_of_freedom(False)
        stargazer.show_model_numbers(False)

        # Get HTML output and add to collection
        html_output = stargazer.render_html()

        # Add section header and content to the collection
        all_html_content.append(f"<h2>{dep_var} - {group_names[group_idx]}</h2>")
        all_html_content.append(html_output)

        # Display in notebook for preview
        display(HTML(html_output))

# Run models for each dependent variable
for dep_var in ["numHelped", "has_helped", "ln_numHelped"]:
    run_models_for_dependent_var(dep_var)

print("All models completed!")

# Create a complete HTML document with all tables
combined_html = f"""
<!DOCTYPE html>
<html>
<head>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            margin-bottom: 20px;
            font-size: 12px;
        }}
        th, td {{
            padding: 8px;
            text-align: center;
            border-bottom: 1px solid #ddd;
        }}
        th {{
            background-color: #f2f2f2;
        }}
        .title {{
            font-size: 14px;
            font-weight: bold;
            margin-bottom: 10px;
            text-align: center;
        }}
        h2 {{
            page-break-before: always;
            margin-top: 20px;
            font-size: 16px;
            color: #333;
            border-bottom: 1px solid #999;
            padding-bottom: 5px;
        }}
        h2:first-child {{
            page-break-before: avoid;
        }}
        .stargazer-notes {{
            font-size: 10px;
            margin-top: 10px;
        }}
    </style>
</head>
<body>
    <h1>Regression Analysis Results</h1>
    {"".join(all_html_content)}
</body>
</html>
"""

# Save the combined HTML to a file
with open('regression_tables.html', 'w') as f:
    f.write(combined_html)

# Convert the HTML to PDF
try:
    # Set options for better PDF output
    options = {
        'page-size': 'Letter',
        'margin-top': '0.75in',
        'margin-right': '0.75in',
        'margin-bottom': '0.75in',
        'margin-left': '0.75in',
        'encoding': 'UTF-8',
        'no-outline': None,
        'enable-local-file-access': None
    }

    pdfkit.from_file('regression_tables.html', 'regression_tables.pdf', options=options)
    print("All tables saved to regression_tables.pdf")
except Exception as e:
    print(f"Error generating PDF: {e}")
    print("HTML version saved as regression_tables.html")

# Clean up the temporary HTML file (optional)
# os.remove('regression_tables.html')

Created numHelpProvidedAT_binned with distribution:
numHelpProvidedAT_binned
0     4237082
5+    1577826
1      569282
2      300302
3      200048
4      148308
Name: count, dtype: int64

Running models for numHelped

Running No Fixed Effects models...
Fitting model with formula: numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)
Model fitted successfully
Fitting model with formula: numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)+ C(numHelpProvidedAT_binned) + C(phase):C(numHelpProvidedAT_binned) + C(has_answer):C(numHelpProvidedAT_binned) + C(phase):C(has_answer):C(numHelpProvidedAT_binned)


In [11]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Define the formula for the three models
formula1 = "numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
formula2 = "user_fe_numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"
formula3 = "question_fe_numHelped ~ C(phase) + C(has_answer) + C(phase):C(has_answer)"

# Model names
model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

# Run and display each model individually
for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"\n\n==== {name} ====")

    # Fit the model
    model = smf.ols(formula=formula, data=df).fit()

    # Apply clustered standard errors by user_id
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=df['user_id']
    )

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Receiving Answers on Providing Help - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))



==== Base Model ====




==== User FE ====




==== Question FE ====


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '
